# Tome 1, Chapter 9: Diophantine Equations (Python companion)

Exercises first, solutions at the end. Computer-science threads: **enumerating solutions** by parametrisation and by
trees (Berggren, Markov) rather than by brute force, a **search** that supports the theorems on Mordell equations,
and **exact rational rotations**, a use of the chord method in computer graphics.

In [ ]:
import math
from fractions import Fraction
from collections import deque
import matplotlib.pyplot as plt

## Exercises

**P9.1 (★) Pythagorean triples, two ways.** List the primitive triples with $c\le N$ by brute force and by Theorem 9.1, check they agree for $N=2000$, and compare running times. Then check numerically that their number is about $\frac N{2\pi}$.

In [ ]:
# Your code here


**P9.2 (★★) The Berggren tree.** The three matrices
$A=\begin{pmatrix}1&-2&2\\2&-1&2\\2&-2&3\end{pmatrix}$, $B=\begin{pmatrix}1&2&2\\2&1&2\\2&2&3\end{pmatrix}$, $C=\begin{pmatrix}-1&2&2\\-2&1&2\\-2&2&3\end{pmatrix}$
send a primitive triple $(a,b,c)^{\mathsf T}$ to three larger ones. Check that a breadth-first search from $(3,4,5)$ produces every primitive triple with $c\le10^4$ exactly once (up to swapping legs).

*How to think about it:* these matrices preserve the quadratic form $a^2+b^2-c^2$ (check $M^{\mathsf T}JM=J$ with $J=\operatorname{diag}(1,1,-1)$), so they map solutions to solutions; the inverse maps decrease $c$, a descent exactly like Markov's.

In [ ]:
# Your code here


**P9.3 (★★) Mordell equations.** For $-30\le k\le30$, $k\ne0$, search the integer solutions of $y^2=x^3+k$ with $y\ge0$ and $|x|\le10^5$. Confirm that $k=7$ and $k=-5$ have none (Theorem 9.5, Exercise 9.4), and find the $k$ with the most solutions.

In [ ]:
# Your code here


**P9.4 (★) The Markov tree.** Generate all Markov triples with largest element $\le10^{12}$ by moves from $(1,1,1)$, and check the uniqueness conjecture (the largest element determines the triple) in that range. Plot $\ln$ of the Markov numbers against their rank.

In [ ]:
# Your code here


**P9.5 (★★) CS application: exact rotations.** Rotating a shape many times by a float angle accumulates rounding errors. Instead, use the chord parametrisation $\cos\theta=\frac{1-t^2}{1+t^2}$, $\sin\theta=\frac{2t}{1+t^2}$ with rational $t\approx\tan\frac\theta2$: the rotation matrix is then exactly orthogonal over $\Q$. Compare the drift of $\|v\|$ after $2000$ rotations of $v=(1,0)$ in floats and in `Fraction`s.

*How to think about it:* a rational point on the unit circle gives a rotation matrix with rational entries whose columns are orthonormal *exactly*, so lengths are preserved exactly; only the angle is approximated, once, at the start.

In [ ]:
# Your code here


---
## Solutions

### P9.1

In [ ]:
import time

def brute(N):
    out = set()
    for c in range(1, N + 1):
        for a in range(1, c):
            b2 = c * c - a * a
            b = math.isqrt(b2)
            if b * b == b2 and a < b and math.gcd(a, b) == 1:
                out.add((a, b, c))
    return out

def param(N):
    out = set()
    m = 2
    while m * m + 1 <= N:
        for n in range(1, m):
            if (m - n) % 2 and math.gcd(m, n) == 1 and m * m + n * n <= N:
                a, b = m * m - n * n, 2 * m * n
                out.add((min(a, b), max(a, b), m * m + n * n))
        m += 1
    return out

t0 = time.time(); B = brute(2000); t1 = time.time(); P = param(2000); t2 = time.time()
print(B == P, len(P), f"brute {t1 - t0:.2f}s, parametrised {t2 - t1:.4f}s")
for N in [10**4, 10**5, 10**6]:
    print(N, len(param(N)), round(N / (2 * math.pi)))

### P9.2

In [ ]:
A = ((1, -2, 2), (2, -1, 2), (2, -2, 3))
Bm = ((1, 2, 2), (2, 1, 2), (2, 2, 3))
Cm = ((-1, 2, 2), (-2, 1, 2), (-2, 2, 3))
mul = lambda M, v: tuple(sum(M[i][j] * v[j] for j in range(3)) for i in range(3))

LIM = 10**4
seen, q = [], deque([(3, 4, 5)])
while q:
    v = q.popleft()
    seen.append(v)
    for M in (A, Bm, Cm):
        w = mul(M, v)
        if w[2] <= LIM:
            q.append(w)
norm = [(min(a, b), max(a, b), c) for a, b, c in seen]
print(len(norm), len(set(norm)), set(norm) == param(LIM))

### P9.3
For each $x$ we test whether $x^3+k$ is a square; `math.isqrt` makes this exact.

In [ ]:
X = 10**5
sols = {k: [] for k in range(-30, 31) if k}
for x in range(-31, X + 1):
    x3 = x ** 3
    for k in sols:
        v = x3 + k
        if v >= 0:
            r = math.isqrt(v)
            if r * r == v:
                sols[k].append((x, r))
print("k = 7:", sols[7], "  k = -5:", sols[-5])
print("no solutions for k in", [k for k in sols if not sols[k]])
kmax = max(sols, key=lambda k: len(sols[k]))
print("most solutions:", kmax, sols[kmax])

### P9.4

In [ ]:
LIM = 10**12
start = (1, 1, 1)
seen, q = {start}, deque([start])
while q:
    t = q.popleft()
    for i in range(3):
        y, z = [t[j] for j in range(3) if j != i]
        new = tuple(sorted((3 * y * z - t[i], y, z)))
        if new[2] <= LIM and new not in seen:
            seen.add(new); q.append(new)
by_max = {}
for t in seen:
    by_max.setdefault(t[2], []).append(t)
print(len(seen), "triples; uniqueness holds:", all(len(v) == 1 for v in by_max.values()))
ms = sorted(by_max)
plt.plot(range(len(ms)), [math.log(m) for m in ms], ".")
plt.xlabel("rank"); plt.ylabel("ln(Markov number)"); plt.show()

Zagier (1982) showed that the number of Markov numbers up to $X$ is about $c(\ln X)^2$; the plot of $\ln m$ against rank is accordingly close to a square-root curve.

### P9.5

In [ ]:
theta = 0.001
t = Fraction(math.tan(theta / 2)).limit_denominator(10**4)
c, s = (1 - t * t) / (1 + t * t), 2 * t / (1 + t * t)
assert c * c + s * s == 1
vx, vy = Fraction(1), Fraction(0)
fx, fy = 1.0, 0.0
cf, sf = math.cos(theta), math.sin(theta)
for _ in range(2000):
    vx, vy = c * vx - s * vy, s * vx + c * vy
    fx, fy = cf * fx - sf * fy, sf * fx + cf * fy
print("exact: |v|^2 - 1 =", vx * vx + vy * vy - 1)
print("float: |v|^2 - 1 =", fx * fx + fy * fy - 1)
print("size of the exact numerator:", vx.numerator.bit_length(), "bits")

The exact rotation keeps $\|v\|=1$ exactly, but its fractions grow (here to tens of thousands of bits, and every step costs more than the last): exactness costs memory. In practice one renormalises floats, or works with Pythagorean angles of small height.